# Lezione 7E — Esercitazione: Agno

**Corso**: Programmazione di Applicazioni Intelligenti  
**Tipo**: Esercitazione (1.5 ore)  
**Prerequisiti**: Notebook 7B (Context Engineering e Agno)

---

**Istruzioni**: completa le celle `# TODO`. Le soluzioni sono nel notebook **7F**.

> Questi esercizi usano **Mistral** tramite Agno. Serve la `MISTRAL_API_KEY` nei Secrets di Colab (gratuita da [console.mistral.ai](https://console.mistral.ai)).


In [ ]:
# === Setup (NON MODIFICARE) ===

!pip install -q agno[mistral,sqlite] ddgs wikipedia

import os
from google.colab import userdata
os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")

print("Setup completato!")


---
## Esercizio 1 — Primo agente Agno (15 min)

**Obiettivo**: creare un agente Agno minimale con:
- Modello: `mistral-small-latest` via `MistralChat`
- Un tool custom: `calculate(expression)` — calcolatrice
- System prompt personalizzato
- Test con: `agent.print_response("Quanto fa 847 * 23 + 156?")`


In [ ]:
# === TODO: Primo agente Agno ===

from agno.agent import Agent
from agno.models.mistral import MistralChat

# TODO: definisci la funzione calculate(expression: str) -> str
# Suggerimento: usa eval() con filtro sui caratteri ammessi


# TODO: crea l'agente con Agent(model=..., tools=[...], instructions=[...], markdown=True)


# TODO: testa con agent.print_response("Quanto fa 847 * 23 + 156?", stream=True)



---
## Esercizio 2 — Progetta un assistente con tool e memoria (25 min)

**Obiettivo**: dato uno scenario, *scegli tu* quali tool servono, implementali, e costruisci un chatbot interattivo con memoria.

**Scenario**: sei un tutor che deve aiutare uno studente universitario a organizzare la sua sessione di studio. Lo studente può chiederti cose come: quanto tempo manca all'esame, quanti crediti ha accumulato, un piano di studio, il meteo del giorno (per decidere se studiare in biblioteca o al parco), ecc.

**Parte A — Progettazione e implementazione (15 min)**

1. Scegli **almeno 2 tool custom, ma non più di 4** per questo scenario. Alcuni esempi (ma puoi inventarne altri):
   - `days_until(date)` — calcola i giorni che mancano a una data
   - `calculate(expression)` — calcolatrice generica
   - `get_weather(city)` — meteo simulato
   - `study_planner(hours_available, num_topics)` — suggerisce come dividere il tempo
   - ...qualsiasi altra cosa ti sembri utile!

2. Implementa i tool come funzioni Python con docstring chiare
3. Crea l'agente con:
   - Istruzioni coerenti con lo scenario
   - `add_history_to_context=True` + `db=SqliteDb(...)` per la memoria

**Parte B — Test interattivo (10 min)**

Implementa il loop `while True` con `input()` (come nel 7B) e fai una conversazione di 4-5 turni che dimostri:
- L'uso di almeno 2 tool diversi
- La memoria (l'agente ricorda qualcosa detto in un turno precedente)

> **Non c'è una soluzione unica**: nel notebook 7F troverai *una* possibile implementazione, ma la tua potrebbe essere diversa e ugualmente valida. L'importante è che i tool siano coerenti con lo scenario e l'agente funzioni.

In [ ]:
# === TODO Parte A: Progetta e implementa il tuo assistente ===

from agno.agent import Agent
from agno.models.mistral import MistralChat
from agno.db.sqlite import SqliteDb

# TODO: scegli e implementa almeno 2 tool per lo scenario "tutor universitario"
# Ricorda: ogni tool deve avere type hints e una docstring descrittiva




# TODO: crea l'agente con i tuoi tool, istruzioni coerenti con lo scenario, e memoria




In [ ]:
# === TODO Parte B: Loop interattivo ===

# TODO: implementa il loop interattivo
# while True:
#     user_input = input("Tu: ")
#     if user_input.strip().lower() in ("esci", "quit", "exit", ""):
#         print("Sessione terminata.")
#         break
#     try:
#         response = agent.run(user_input)
#         print(f"Tutor: {response.content}\n")
#     except Exception as e:
#         print(f"Errore: {e}\n")



> **Dopo il test**: in una cella markdown sotto, scrivi 2-3 righe che spiegano:
> 1. Quali tool hai scelto e perché
> 2. La memoria ha funzionato? Come l'hai verificato?

---
## Esercizio 3 — Artigianale vs Framework: confronto ragionato (15 min)

**Obiettivo**: confrontare il loop ReAct del notebook 7C con l'agente Agno, sottomettendo la **stessa domanda** a entrambi, e riflettere sulle differenze.

**Cosa fare**:

1. Nella cella sotto trovi il loop ReAct artigianale del 7C, con 3 tool: `search_wikipedia`, `calculate`, `get_current_date`
2. Crea un agente Agno equivalente: stesso modello (`mistral-small-latest`), **stessi 3 tool**
3. Sottometti la **stessa domanda** a entrambi: **"In che anno è nato Leonardo da Vinci e quanti anni fa è stato?"**

> **Importante**: usa la stessa domanda, lo stesso modello (`mistral-small-latest`) e gli stessi tool (`search_wikipedia` + `calculate` + `get_current_date`) in entrambi gli approcci — altrimenti il confronto non è valido.

4. Confronta:
   - Entrambi arrivano alla risposta corretta?
   - Quanti step/tool call fa ciascuno?
   - Quanto codice hai scritto per l'uno e per l'altro?
5. Scrivi un breve commento (3-4 righe in una cella markdown) su vantaggi e svantaggi di ciascun approccio

In [ ]:
# === Approccio 1: loop ReAct artigianale (dal 7C) ===
# Questa funzione è identica a quella del notebook 7C — NON MODIFICARE

from openai import OpenAI
from google.colab import userdata
from datetime import datetime
import json

client_oai = OpenAI(
    base_url="https://api.mistral.ai/v1",
    api_key=userdata.get("MISTRAL_API_KEY"),
)

def search_wikipedia(query: str) -> str:
    """Cerca informazioni su Wikipedia. Restituisce un breve estratto."""
    knowledge = {
        "leonardo da vinci": "Leonardo di ser Piero da Vinci (1452-1519) e' stato un inventore, artista e scienziato italiano del Rinascimento. Nato a Vinci, in Toscana, il 15 aprile 1452.",
        "albert einstein": "Albert Einstein (1879-1955) e' stato un fisico teorico tedesco. Nato a Ulm il 14 marzo 1879. Premio Nobel per la fisica nel 1921.",
    }
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return value
    return f"Nessun risultato trovato per: {query}"

def calculate(expression: str) -> str:
    """Calcola un'espressione matematica."""
    allowed = set("0123456789+-*/.(). ")
    if not all(c in allowed for c in expression):
        return "Errore: espressione non valida"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Errore: {e}"

def get_current_date() -> str:
    """Restituisce la data corrente."""
    now = datetime.now()
    return f"Oggi e' il {now.day}/{now.month}/{now.year}"

tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "search_wikipedia",
            "description": "Cerca informazioni su Wikipedia. Restituisce un breve estratto.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Termine di ricerca"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Calcola un'espressione matematica.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Espressione matematica"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "Restituisce la data corrente.",
            "parameters": {"type": "object", "properties": {}}
        }
    }
]

tool_registry = {
    "search_wikipedia": search_wikipedia,
    "calculate": calculate,
    "get_current_date": get_current_date,
}

def react_agent(question, tools, tool_registry, max_steps=10):
    messages = [
        {"role": "system", "content": "Sei un assistente. Usa i tool per rispondere. Cerca sempre le informazioni con search_wikipedia prima di rispondere."},
        {"role": "user", "content": question}
    ]
    for step in range(max_steps):
        response = client_oai.chat.completions.create(
            model="mistral-small-latest", messages=messages, tools=tools
        )
        msg = response.choices[0].message
        if response.choices[0].finish_reason == "stop":
            return msg.content
        if response.choices[0].finish_reason == "tool_calls":
            messages.append(msg)
            for tc in msg.tool_calls:
                fn = tool_registry[tc.function.name]
                args = json.loads(tc.function.arguments)
                result = fn(**args) if args else fn()
                print(f"  [ReAct] {tc.function.name}({args}) -> {result}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
    return "Timeout: troppi step"

# === Test ReAct artigianale ===
DOMANDA = "In che anno e' nato Leonardo da Vinci e quanti anni fa e' stato?"
print("=== APPROCCIO 1: ReAct artigianale ===")
risposta_react = react_agent(DOMANDA, tools_schema, tool_registry)
print(f"Risposta: {risposta_react}")


In [ ]:
# === TODO: Approccio 2 — stesso task con Agno ===

from agno.agent import Agent
from agno.models.mistral import MistralChat

# I tool search_wikipedia(), calculate() e get_current_date() sono gia' definiti sopra

# TODO: crea un agente Agno con gli STESSI 3 tool e lo stesso modello
# Suggerimento: tools=[search_wikipedia, calculate, get_current_date]


# TODO: testa con la stessa domanda
# print("\n=== APPROCCIO 2: Agno ===")
# agent.print_response(DOMANDA, stream=True)



**TODO**: scrivi qui il tuo confronto (3-4 righe):

- Entrambi arrivano alla risposta corretta?
- Entrambi usano `search_wikipedia` per trovare l'anno di nascita, o uno dei due "sa già" la risposta?
- Quanti tool call fa ciascuno?
- Quanto codice hai scritto per ciascun approccio?
- Quale preferiresti per un progetto reale? Perché?

---
## Esercizio 4 — Multi-tool con toolkit Agno: Tech News Analyst (25 min)

**Obiettivo**: costruire un agente che combina **due toolkit Agno preconfezionati** con un **tool custom**, per analizzare le notizie tech del momento.

L'agente deve:
1. Usare `HackerNewsTools` per recuperare le top story da Hacker News
2. Usare `WikipediaTools` per cercare contesto su un argomento menzionato nelle notizie
3. Usare un **tool custom** `classify_topic(title: str) -> str` che classifica il titolo di una notizia in una categoria (AI, Web, Security, Hardware, Business, Other)
4. Sintetizzare un breve "bollettino tech" in italiano

**Requisiti**:
- Installa e importa: `from agno.tools.hackernews import HackerNewsTools` e `from agno.tools.wikipedia import WikipediaTools`
- Il tool custom `classify_topic` deve usare keyword matching (no LLM): se il titolo contiene "AI", "LLM", "GPT", "model" → `"AI"`, ecc.
- L'agente usa tutti e 3 i tool insieme
- Testa con: `"Recupera le top 3 story da Hacker News, classifica ciascuna per categoria, poi cerca su Wikipedia il tema della prima story e scrivi un bollettino tech in italiano."`

**Cosa impari**: come comporre toolkit built-in di Agno con tool custom nello stesso agente; l'agente decide autonomamente quale tool usare e in che ordine.

> **Suggerimento**: `HackerNewsTools()` espone `get_top_hackernews_stories(num_stories)` e `WikipediaTools()` espone `search_wikipedia(query)`.


In [ ]:
# === TODO: Tech News Analyst ===

from agno.agent import Agent
from agno.models.mistral import MistralChat
from agno.tools.hackernews import HackerNewsTools
from agno.tools.wikipedia import WikipediaTools

# TODO: implementa classify_topic(title: str) -> str
# Classificazione basata su keyword matching:
#   - AI/LLM/GPT/model/neural/transformer -> "AI"
#   - web/browser/CSS/HTML/JavaScript/React -> "Web"
#   - security/hack/vulnerability/CVE/encrypt -> "Security"
#   - chip/CPU/GPU/hardware/RISC/ARM -> "Hardware"
#   - startup/funding/acquisition/IPO/revenue -> "Business"
#   - altrimenti -> "Other"



# TODO: crea l'agente con tools=[HackerNewsTools(), WikipediaTools(), classify_topic]
# Istruzioni suggerite:
#   - "Sei un analista tech. Scrivi sempre in italiano."
#   - "Usa get_top_hackernews_stories per le notizie."
#   - "Usa classify_topic per classificare ogni titolo."
#   - "Usa search_wikipedia per approfondire un argomento."



# TODO: testa con:
# agent.print_response(
#     "Recupera le top 3 story da Hacker News, classifica ciascuna per categoria, "
#     "poi cerca su Wikipedia il tema della prima story e scrivi un bollettino tech in italiano.",
#     stream=True
# )


---
## Esercizio 3bis (facoltativo) — Agente ibrido: web search + tool custom

**Obiettivo**: creare un agente che combina **web search reale** con **tool custom** per rispondere a domande che richiedono sia dati dal web sia elaborazioni locali.

Esempio di task: "Cerca quanti abitanti ha Roma e quanti ne ha Milano, poi calcola la differenza."

L'agente deve:
1. Usare `WebSearchTools` per cercare i dati reali
2. Usare `calculate()` per fare il calcolo
3. Sintetizzare il risultato

Requisiti:
- Un unico agente con **entrambi** i tool: `WebSearchTools()` e `calculate`
- Istruzioni che spieghino all'agente di usare la ricerca per i dati e il calcolo per le elaborazioni


> **Da fare solo se il gruppo è veloce, oppure come homework.** Non è necessario completarlo in aula per proseguire nel corso.

In [ ]:
# === TODO: Agente ibrido web search + custom tool ===

from agno.agent import Agent
from agno.models.mistral import MistralChat
from agno.tools.websearch import WebSearchTools

# TODO: definisci calculate(expression: str) -> str (riuso dai precedenti)


# TODO: crea l'agente con tools=[WebSearchTools(), calculate]
# Nelle istruzioni, specifica che deve usare web search per dati reali
# e calculate per le elaborazioni numeriche


# TODO: testa con domande che richiedono entrambi i tool, ad esempio:
# "Cerca la popolazione di Roma e di Milano, poi calcola la differenza"
# oppure: "Cerca la distanza Roma-Milano in km e calcola quanto tempo ci vuole a 130 km/h"

